<a href="https://colab.research.google.com/github/hemanthvnp/Linear-Discriminant-Analysis/blob/main/notebooks/colab_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MvDA on Colab

Run Multi-view Discriminant Analysis on Colab, reading the **ColorFERET** images directly from your Google Drive (no manual download).

If you upload a zip of the repo instead of using GitHub, skip cell 1 and `cd` into the unzipped folder.

## 1. Get the code

In [1]:
!git clone https://github.com/hemanthvnp/Linear-Discriminant-Analysis.git mvda
%cd mvda
!pip -q install -r requirements.txt

Cloning into 'mvda'...
remote: Enumerating objects: 40, done.
remote: Counting objects: 100% (40/40), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 40 (delta 1), reused 40 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (40/40), 30.48 KiB | 7.62 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/mvda


## 2. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 3. Locate the ColorFERET folder and confirm it has images

This finds the folder and checks for actual image files. If the image count is 0, your Drive copy only has metadata (like the original local copy) and you'll need the real `.ppm`/`.ppm.bz2` images.

In [3]:
!find /content/drive/MyDrive -maxdepth 4 -iname '*colorferet*' -type d 2>/dev/null | head
FERET_ROOT = '/content/drive/MyDrive/colorferet'  # <-- set to the path printed above
!echo 'image files found:'; find "$FERET_ROOT" -type f \( -iname '*.ppm' -o -iname '*.ppm.bz2' -o -iname '*.png' -o -iname '*.jpg' \) 2>/dev/null | wc -l

/content/drive/MyDrive/colorferet
/content/drive/MyDrive/colorferet/colorferet
image files found:
50174


## 4. Sanity check on UCI Multiple Features (auto-downloads, no Drive needed)

In [4]:
!python experiments/run_mvda.py --dataset mfeat --mode concat --classifier ensemble

Loading dataset=mfeat scaler=robust ...
  views=6 dims=[76, 216, 64, 240, 47, 6] train=1000 test=1000 classes=10
Fitting MultiViewLDA(mode=concat, vc_lambda=0.0) ...
  shared-space dim = 9

=== mfeat | concat | ensemble ===
Accuracy: 98.700%

class   precision   recall      f1          support   
------------------------------------------------------
0       1.0000      0.9900      0.9950      100       
1       0.9709      1.0000      0.9852      100       
2       0.9900      0.9900      0.9900      100       
3       0.9796      0.9600      0.9697      100       
4       0.9901      1.0000      0.9950      100       
5       0.9898      0.9700      0.9798      100       
6       1.0000      0.9900      0.9950      100       
7       0.9804      1.0000      0.9901      100       
8       1.0000      0.9900      0.9950      100       
9       0.9703      0.9800      0.9751      100       
------------------------------------------------------
macro   0.9871      0.9870      0.9870    

## 5. Run MvDA on ColorFERET

`--feret-poses` picks which poses become views (subjects missing any are dropped). Start small with `--feret-max-subjects` for a quick first run, then scale up.

In [5]:
!python experiments/run_mvda.py --dataset colorferet --mode mvda --classifier ncm \
    --feret-root "$FERET_ROOT" --feret-poses ql fa qr --feret-size 64 64 \
    --feret-max-subjects 100 --save feret_mvda.json

Loading dataset=colorferet scaler=robust ...
Traceback (most recent call last):
  File "/content/mvda/experiments/run_mvda.py", line 76, in <module>
    main()
  File "/content/mvda/experiments/run_mvda.py", line 51, in main
    Xtr, Xte, ytr, yte = load_dataset(args)
                         ^^^^^^^^^^^^^^^^^^
  File "/content/mvda/experiments/_common.py", line 55, in load_dataset
    Xtr, Xte, ytr, yte = _split_per_class(views, y, args.train_frac, args.seed)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/mvda/experiments/_common.py", line 38, in _split_per_class
    return ([v[train_idx] for v in views], [v[test_idx] for v in views],
                                            ~^^^^^^^^^^
IndexError: arrays used as indices must be of integer (or boolean) type


## 6. (optional) Ablations and cross-validation

In [6]:
!python experiments/per_view_analysis.py --dataset colorferet --feret-root "$FERET_ROOT"
!python experiments/ablation_components.py --dataset colorferet --feret-root "$FERET_ROOT" --mode mvda

Traceback (most recent call last):
  File "/content/mvda/experiments/per_view_analysis.py", line 50, in <module>
    main()
  File "/content/mvda/experiments/per_view_analysis.py", line 30, in main
    Xtr, Xte, ytr, yte = load_dataset(args)
                         ^^^^^^^^^^^^^^^^^^
  File "/content/mvda/experiments/_common.py", line 55, in load_dataset
    Xtr, Xte, ytr, yte = _split_per_class(views, y, args.train_frac, args.seed)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/mvda/experiments/_common.py", line 38, in _split_per_class
    return ([v[train_idx] for v in views], [v[test_idx] for v in views],
                                            ~^^^^^^^^^^
IndexError: arrays used as indices must be of integer (or boolean) type
Traceback (most recent call last):
  File "/content/mvda/experiments/ablation_components.py", line 42, in <module>
    main()
  File "/content/mvda/experiments/ablation_components.py", line 26, in main
   